# AHỤIKE 02 — QLoRA fine-tuning of N-ATLaS-8B (Unsloth)
Trains **AHỤIKE** (parallel-anchored) and the **ablation** (same size, no parallel anchoring). Each run takes about 1.5–3 h on one T4. Checkpoints every 50 steps: if the session dies, re-run with `--resume`.

Kaggle settings: **GPU T4 x2**, **Internet on**, Secret **HF_TOKEN** attached.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/EgwuSamuel/ahuike.git ahuike || (cd ahuike && git pull -q)
%cd ahuike

Pre-flight check: token, gated access to N-ATLaS + 4 ASR models, GPUs, chat template. Everything should say PASS before you continue.

In [ ]:
!python scripts/check_setup.py

In [ ]:
!pip install -q unsloth
!pip uninstall -y -q torchaudio  # avoid CUDA-version mismatch with the new torch

In [ ]:
!python scripts/hub_sync.py pull data

## 1. AHỤIKE (parallel-anchored)

In [ ]:
!python scripts/finetune.py --data data/sft_anchored.jsonl --out outputs/ahuike-lora --epochs 2

In [ ]:
!python scripts/hub_sync.py push outputs/ahuike-lora

## 2. Ablation (no parallel anchoring) — run if time allows

In [ ]:
!python scripts/finetune.py --data data/sft_ablation.jsonl --out outputs/ablation-lora --epochs 2

In [ ]:
!python scripts/hub_sync.py push outputs/ablation-lora

## 3. Export merged Q4_K_M GGUF for the CPU demo (HF Space)
Repo name must carry the **Powered-by-Awarri** suffix (N-ATLaS licence).

In [ ]:
HF_USER = 'SamEgwu'
!python scripts/export_gguf.py --adapter outputs/ahuike-lora --out /tmp/ahuike-gguf --push {HF_USER}/AHUIKE-N-ATLaS-8B-GGUF-Powered-by-Awarri